# 03 · A mechanistic model from scratch (ODEs, RK4, state reconstruction)

**Goal:** write the glucose–insulin ODE yourself, integrate it, understand its behaviour as a
filter, use it to forecast, and see what fitting it teaches about identifiability.

**Time:** ~1.5 h · **You'll write:** `rk4_step`, `minimal_model_rhs`, `forecast_ode`

**How the exercises work.** Each one is a function with `raise NotImplementedError`. Replace that
with your code, then run the cell below it. `use(...)` checks your function against an independent
reference (the project's own code, autograd, or a closed form):
- `[ok]` → your version is used from then on.
- `[todo]` / `[fail]` → the notebook falls back to the reference solution so you can keep going.

Hints are folded under each exercise. Solutions are in `tutorials/solutions.py`; try first.

In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == "tutorials":
    os.chdir("..")
sys.path.insert(0, "tutorials")

import numpy as np
import torch
import matplotlib.pyplot as plt
from tutor import use, close
import solutions as sol

torch.set_num_threads(4)
plt.rcParams.update({"figure.figsize": (10, 3.2), "axes.grid": True, "grid.alpha": 0.3})

from biotwin.data.dataset import Cohort, standard_sets, subset, LH, H
from biotwin.models.mechanistic import MinimalModel, PARAM_NAMES, INIT
from biotwin.train import fit, predict
cohort = Cohort.load("data/cohort.npz")
S = standard_sets(cohort, seed=0)

## 1. Integrating an ODE: RK4

A model $\dot x = f(x)$ only defines a derivative. To simulate it we step forward in time. Euler
($x + h f(x)$) has error $O(h)$. Classic **Runge–Kutta 4** samples the slope four times per step
and has error $O(h^4)$:

$$k_1 = f(x),\quad k_2 = f(x + \tfrac h2 k_1),\quad k_3 = f(x + \tfrac h2 k_2),\quad k_4 = f(x + h k_3)$$
$$x_{\text{next}} = x + \tfrac h6 (k_1 + 2k_2 + 2k_3 + k_4)$$

## Exercise 1 — `rk4_step(f, x, h)`


<details><summary><b>Hint</b></summary>

Four lines for k1..k4, one for the update. It works for any tensor shape as long as `f` does.

</details>

In [ ]:
def rk4_step(f, x, h):
    # TODO
    raise NotImplementedError

def check(f_):
    x = torch.tensor([1.0, 2.0], dtype=torch.float64)
    close(f_(lambda z: -z, x, 0.1), x * np.exp(-0.1), tol=1e-7, what="one step of dx/dt = -x")

rk4_step = use(rk4_step, sol.rk4_step, check)

In [ ]:
# convergence: integrate dx/dt = -x from 0 to 1 with step h; error vs exact e^-1
hs = [0.5, 0.25, 0.1, 0.05, 0.02]
err_rk4, err_eul = [], []
for h in hs:
    x4 = xe = torch.tensor(1.0, dtype=torch.float64)
    for _ in range(round(1 / h)):
        x4 = rk4_step(lambda z: -z, x4, h)
        xe = xe + h * (-xe)
    err_rk4.append(abs(x4.item() - np.exp(-1))); err_eul.append(abs(xe.item() - np.exp(-1)))
plt.figure(figsize=(5, 3.5))
plt.loglog(hs, err_eul, "o-", label="Euler"); plt.loglog(hs, err_rk4, "o-", label="RK4")
plt.xlabel("step h"); plt.ylabel("error at t=1"); plt.legend(); plt.show()
print("slopes (≈ order):", np.polyfit(np.log(hs), np.log(err_eul), 1)[0].round(2), np.polyfit(np.log(hs), np.log(err_rk4), 1)[0].round(2))

## 2. The minimal model

Seven states, with inputs $u$ = insulin above basal (U/min) and $r$ = carbs eaten (g/min):

$$
\begin{aligned}
\dot I_{sc1} &= u - k_a I_{sc1} & \dot I_{sc2} &= k_a (I_{sc1} - I_{sc2}) & \dot I_p &= k_a I_{sc2} - k_e I_p\\
\dot X &= p_2 (S_I I_p - X) & \dot D_1 &= r - k_d D_1 & \dot D_2 &= k_d (D_1 - D_2)\\
\dot G &= -S_G (G - G_b) - X G + c_g k_d D_2
\end{aligned}
$$

The state order is $[I_{sc1}, I_{sc2}, I_p, X, D_1, D_2, G]$ and the parameter order is
$\theta = [k_a, k_e, k_d, S_G, p_2, S_I, c_g, G_b]$.

## Exercise 2 — the right-hand side

`s` is `[B, 7]`, `u` and `r` are `[B]`, `th` is `[B, 8]`. Return `ds/dt`, `[B, 7]`.


<details><summary><b>Hint</b></summary>

`ka, ke, ... = th.unbind(-1)` and `Isc1, ..., G = s.unbind(-1)`, then `torch.stack([...], -1)`.

</details>

In [ ]:
def minimal_model_rhs(s, u, r, th):
    # TODO
    raise NotImplementedError

def check(f):
    m = MinimalModel(); g = torch.Generator().manual_seed(0)
    s, u, r = 10 * torch.rand(6, 7, generator=g), torch.rand(6, generator=g), torch.rand(6, generator=g)
    th = m.log_theta.exp().expand(6, -1)
    close(f(s, u, r, th), m.f(s, u, r, th, None), what="ds/dt")

minimal_model_rhs = use(minimal_model_rhs, sol.minimal_model_rhs, check)

## 3. What the model does: responses to a meal and to a bolus

We start at steady state (no extra insulin or carbs, $G = G_b$) with textbook parameter values,
and simulate 6 h at 1-minute steps.

In [ ]:
th0 = torch.tensor([INIT[k] for k in PARAM_NAMES])[None]

def simulate(u_fn, r_fn, G0=140.0, minutes=360, th=th0):
    th = th.clone(); th[0, 7] = G0                 # steady state at G0
    s = torch.zeros(1, 7); s[0, 6] = G0
    out = [s]
    for t in range(minutes):
        u, r = torch.tensor([u_fn(t)]), torch.tensor([r_fn(t)])
        s = rk4_step(lambda z: minimal_model_rhs(z, u, r, th), s, 1.0)
        out.append(s)
    return torch.cat(out).numpy()

meal = simulate(lambda t: 0.0, lambda t: 5.0 if t < 10 else 0.0)     # 50 g over 10 min
bolus = simulate(lambda t: 3.0 if t < 1 else 0.0, lambda t: 0.0)      # 3 U at t=0
both = simulate(lambda t: 3.0 if t < 1 else 0.0, lambda t: 5.0 if t < 10 else 0.0)
tm = np.arange(361) / 60
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
axes[0].plot(tm, meal[:, 4], label="D1 (stomach)"); axes[0].plot(tm, meal[:, 5], label="D2 (gut)"); axes[0].set_title("carbs through the gut")
axes[1].plot(tm, bolus[:, 0], label="Isc1"); axes[1].plot(tm, bolus[:, 2], label="Ip"); axes[1].plot(tm, bolus[:, 3] * 100, label="X ×100"); axes[1].set_title("insulin chain")
axes[2].plot(tm, meal[:, 6], label="meal only"); axes[2].plot(tm, bolus[:, 6], label="bolus only"); axes[2].plot(tm, both[:, 6], label="meal + bolus"); axes[2].set_title("glucose")
for a in axes: a.legend(); a.set_xlabel("hours")
plt.tight_layout(); plt.show()

Each compartment is a smoother, later copy of the one before: a cascade of first-order low-pass
filters. Now the one nonlinearity. The same 3 U bolus, given to two patients sitting at
different glucose levels:

In [ ]:
for G0 in (100, 250):
    tr_ = simulate(lambda t: 3.0 if t < 1 else 0.0, lambda t: 0.0, G0=G0)
    plt.plot(tm, tr_[:, 6] - G0, label=f"starting at {G0} mg/dL")
plt.ylabel("Δ glucose"); plt.xlabel("hours"); plt.legend(); plt.title("same bolus, different glucose level"); plt.show()

The drop is bigger at 250 because of the $-X\,G$ term: insulin action **multiplies** glucose. A
linear model (ARX) can't do this. With fixed coefficients, the same bolus always has the same
effect.

## 4. Forecasting with hidden states

At forecast time we observe only $G$ (through CGM). The other six states are hidden. They're
linear filters of **known** inputs, so we can reconstruct them by running the model over the
6-hour history. That's an open-loop observer:

1. start from zeros;
2. for each history step $t$: set $G$ to the measured CGM, then integrate one 5-min step with
   that step's inputs;
3. after the history, set $G$ to the last CGM;
4. integrate $H$ more steps with the **planned** inputs, recording $G$, clamped to [20, 600].

Inputs are zero-order held over each 5-min step. Use 2 RK4 substeps of 2.5 min, with
`u = (ins - basal) / 5` and `r = carbs / 5` (per minute).

## Exercise 3 — `forecast_ode(th, w, LH)`


<details><summary><b>Hint</b></summary>

Write a helper `advance(s, t)` that does the 2 RK4 substeps with the inputs at step `t`.
Pinning G looks like `s = torch.cat([s[:, :6], g[:, t:t+1]], 1)`. The clamp only applies during the forecast.

</details>

In [ ]:
def forecast_ode(th, w, LH, substeps=2, step_min=5.0):
    # TODO: th [B, 8] in natural units; return [B, H]
    raise NotImplementedError

def check(f):
    m = MinimalModel(); m.load_state_dict(torch.load("results/checkpoints/ode.pt"))
    w = subset(S["test_eval"], torch.arange(len(S["test_eval"]["y"])) < 64)
    with torch.no_grad():
        close(f(m.log_theta.exp().expand(64, -1), w, LH), m(w), tol=1e-4, what="forecast")

forecast_ode = use(forecast_ode, sol.forecast_ode, check)

## 5. Fitting the 8 parameters

The forecast is differentiable end to end (84 steps × 2 substeps × 4 slope evaluations), so we
fit $\log\theta$ by gradient descent on the forecast error. Fitting logs keeps the parameters
positive. Here's one minute of training from textbook values, on a subset, using the project's
training loop:

In [ ]:
m = MinimalModel()
small_tr = subset(S["train"], torch.arange(len(S["train"]["y"])) % 12 == 0)
small_va = subset(S["val"], torch.arange(len(S["val"]["y"])) % 4 == 0)
m, hist = fit(m, small_tr, small_va, epochs=8, lr=0.03, bs=256, gen=torch.Generator().manual_seed(0),
              max_minutes=1.0, verbose=False)
print("val RMSE by epoch:", [round(h["val_rmse"], 1) for h in hist])

full = MinimalModel(); full.load_state_dict(torch.load("results/checkpoints/ode.pt"))
for k, a, b, c in zip(PARAM_NAMES, [INIT[k] for k in PARAM_NAMES], m.log_theta.exp().tolist(), full.log_theta.exp().tolist()):
    print(f"{k:3s} textbook {a:8.4g} | 1-min fit {b:8.4g} | full fit {c:8.4g}")

## 6. Identifiability

The full fit gives $S_G \approx 0.0025$ and $G_b \approx 79$ mg/dL. That $G_b$ isn't
physiological. To see why the data allows it, map the loss over a grid of $(S_G, G_b)$ with every
other parameter fixed at the fitted values:

In [ ]:
w = subset(S["val"], torch.arange(len(S["val"]["y"])) % 10 == 0)
base = full.log_theta.detach().clone()
SG = np.logspace(-4, -1.3, 14); GB = np.linspace(50, 200, 14)
L = np.zeros((len(GB), len(SG)))
with torch.no_grad():
    for i, gb in enumerate(GB):
        for j, sg in enumerate(SG):
            lt = base.clone(); lt[3] = np.log(sg); lt[7] = np.log(gb)
            L[i, j] = (full(w, log_theta=lt)[:, -1] - w["y"][:, -1]).pow(2).mean().sqrt()
plt.figure(figsize=(6, 4))
cs = plt.contourf(SG, GB, L, levels=20); plt.colorbar(cs, label="val RMSE 60 min")
plt.xscale("log"); plt.xlabel("S_G"); plt.ylabel("G_b"); plt.plot(base[3].exp(), base[7].exp(), "r*", ms=12)
plt.show()

The dark valley is long in the $G_b$ direction. At the fitted $S_G$, anything from about 50 to
160 mg/dL fits almost equally well. Below $S_G \approx 10^{-3}$, $G_b$ doesn't matter at all,
because the term $S_G(G - G_b)$ is tiny. The data can't pin down $G_b$. It's a ridge in the likelihood,
exactly like a poorly excited mode in system identification. Lesson: a good fit doesn't mean
every parameter means something. Check the curvature (notebook 06 does it with a Hessian).

## Quiz

Answer in your own words before opening the answers.

**1. Why does RK4 need 4 evaluations of f per step, and what do you buy with them?**

<details><summary>Answer</summary>

It combines four slope samples so that the error terms cancel up to 4th order: local error O(h⁵), global O(h⁴). Halving h cuts the error ~16×, versus 2× for Euler. In practice you can take much larger steps for the same accuracy.

</details>

**2. In the response plots, why does Ip peak later than Isc1, and X later still?**

<details><summary>Answer</summary>

Each compartment is a first-order low-pass filter fed by the previous one, so every stage adds delay and smoothing (the group delay adds up). X sits after Ip with its own time constant 1/p2.

</details>

**3. What makes the model nonlinear, and what does that mean for a bolus at 100 vs 250 mg/dL?**

<details><summary>Answer</summary>

The −X·G term (bilinear). The same insulin action X removes glucose in proportion to G, so the drop is larger at 250. A linear model would predict the same drop at both levels.

</details>

**4. Why can we reconstruct the hidden states by running the model over the history, and why does that need about 6 h?**

<details><summary>Answer</summary>

The hidden states are outputs of linear filters driven by known inputs (insulin, carbs), so simulating the filters from any initial condition converges to the true trajectory once the initial transient has decayed. The slowest time constants (insulin absorption and action) are tens of minutes to hours, so we need several of them.

</details>

**5. The loss is flat along G_b when S_G is small. What would you do about it?**

<details><summary>Answer</summary>

Options: put a prior on G_b (Bayesian regularisation, which is what notebook 06 does), fix it to a literature value, reparameterise (e.g. fit the product S_G·G_b), or collect data that excites that mode (long fasting periods where glucose relaxes toward G_b). Above all, report it as weakly identifiable and don't interpret the value.

</details>